# 06 - model evaluation (sample)

evaluate svm on sample data: per-class metrics, error analysis.

In [ ]:
import joblib, pandas as pd, numpy as np, librosa
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns, matplotlib.pyplot as plt
from pathlib import Path
from src.config.settings import SAMPLE_RATE, N_MFCC, N_FFT, HOP_LENGTH

SAMPLES = Path("../../data/samples")
df = pd.read_csv(SAMPLES / "sample_labels.csv")
svm = joblib.load(SAMPLES / "svm_baseline_sample.pkl")

In [ ]:
def extract_mfcc(fp):
    y,sr = librosa.load(fp, sr=SAMPLE_RATE)
    m = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH)
    d = librosa.feature.delta(m)
    d2 = librosa.feature.delta(m, order=2)
    return np.concatenate([m.mean(1), m.std(1), d.mean(1), d.std(1), d2.mean(1), d2.std(1)])

test = df[df['split']=='test']
X_test = np.array([extract_mfcc(fp) for fp in test['filepath']])
y_test = test['emotion_code'].values
scaler = StandardScaler().fit(X_test)
y_pred = svm.predict(scaler.transform(X_test))
emap = {1:'neutral',2:'calm',3:'happy',4:'sad',5:'angry',6:'fearful',7:'disgust',8:'surprised'}

In [ ]:
print(classification_report(y_test, y_pred, target_names=list(emap.values()), digits=3))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=list(emap.values()), yticklabels=list(emap.values()))
plt.title('svm confusion (sample test)')
plt.show()
tp = np.trace(cm)
print(f"correct: {tp}/{len(y_test)} ({tp/len(y_test)*100:.1f}%)")

evaluation complete. confusion matrix shows which emotions the model confuses.